# Load the certified gold tables into a Fabric Lakehouse

In production, gold tables would reach Fabric through a **Mirrored Azure Databricks Catalog** (no copy), with OneLake security re-applied on the same Entra ID groups. For this demo, the gold tables exported from the Databricks pipeline (`data/gold/*.parquet` in the GitHub repo) are loaded into a Lakehouse as Delta tables. The Direct Lake semantic model, Power BI report, Power Automate flow and Copilot Studio agent all read these.

**Before running:** attach this notebook to the `copilot_lakehouse` Lakehouse (left pane → Lakehouses → Add).

In [ ]:
# Parameters
GITHUB_RAW = "https://raw.githubusercontent.com/jjaishwin/agency-performance-copilot/main/data/gold"
TABLES = ["account_market_month", "agency_month", "agency_scorecard", "account_scorecard", "contract_scenarios",
          "account_briefs", "dim_account", "dim_market", "aadv_business_compliance", "target_points"]

In [ ]:
import pandas as pd
from pyspark.sql import functions as F

for t in TABLES:
    pdf = pd.read_parquet(f"{GITHUB_RAW}/{t}.parquet")
    df = spark.createDataFrame(pdf)
    for c in ("month", "quarter_start"):
        if c in df.columns:
            df = df.withColumn(c, F.to_date(c))
    df.write.mode("overwrite").format("delta").option("overwriteSchema", "true").saveAsTable(t)
    print(f"{t:<26} {df.count():>7,} rows")

In [ ]:
# Quick check: the same numbers as Databricks
display(spark.sql("""
SELECT risk_status, COUNT(*) AS accounts FROM account_scorecard GROUP BY risk_status ORDER BY accounts DESC
"""))